# Simulador: Ascenso de Colina
### Capítulo 4 — *Aprendizaje y Comportamiento Adaptable: Principios y Modelos*
**Arturo Bouzas** · Facultad de Psicología, UNAM · bouzaslab25.com

---

Una bacteria (*Salmonella*) busca nutrientes en un entorno con múltiples fuentes de concentración
química. Sin receptores de distancia, solo puede comparar la concentración *ahora* con la
concentración *hace un momento*. La ecuación que gobierna su comportamiento:

$$Y(t+1) = a \cdot Y(t) + b \cdot [X(t+1) - X(t)]$$

| Símbolo | Significado |
|---------|-------------|
| $Y(t)$ | Variable de decisión: positiva → explotar, negativa → explorar |
| $X(t)$ | Concentración química en la posición actual |
| $a$ | Adaptación sensorial $(0 < a < 1)$ |
| $b$ | Sensibilidad al cambio $(b > 0)$ |

**Regla de respuesta:**  $\;Y > 0 \Rightarrow$ **Explotar** (nado recto) $\quad$ $Y \leq 0 \Rightarrow$ **Explorar** (maromas aleatorias)


## Celda 1 — Importaciones

In [ ]:
# Importaciones — ejecutar primero
# En Colab ya están instaladas; en Jupyter local puede requerirse:
# pip install ipywidgets matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings("ignore")

# En Colab: activar el gestor de widgets personalizado
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # En Jupyter local no es necesario

WORLD_SIZE = 100
np.random.seed(42)
print("Importaciones completadas.")


## Celda 2 — Paisaje de concentración

Dos fuentes de nutrientes: una principal (la meta) y un máximo local (la trampa).

In [ ]:
# ── Paisaje de concentración ───────────────────────────────────────────────
# El entorno tiene dos fuentes de nutrientes:
#   - Pico principal (meta, arriba a la derecha)
#   - Máximo local  (trampa, abajo a la izquierda)

def build_landscape(world_size=WORLD_SIZE):
    xs = np.linspace(0, 1, world_size)
    ys = np.linspace(0, 1, world_size)
    X, Y = np.meshgrid(xs, ys)

    peaks = [
        dict(cx=0.75, cy=0.75, strength=1.00, sigma=0.18),   # pico principal
        dict(cx=0.22, cy=0.30, strength=0.55, sigma=0.12),   # máximo local (trampa)
        dict(cx=0.55, cy=0.55, strength=0.25, sigma=0.09),   # montículo menor
    ]
    Z = np.zeros((world_size, world_size))
    for p in peaks:
        dx, dy = X - p["cx"], Y - p["cy"]
        Z += p["strength"] * np.exp(-(dx**2 + dy**2) / (2 * p["sigma"]**2))

    Z += 0.04 * X + 0.03 * Y          # gradiente de fondo suave
    Z = (Z - Z.min()) / (Z.max() - Z.min())
    return Z

LANDSCAPE = build_landscape()
CMAP = LinearSegmentedColormap.from_list(
    "nutrientes",
    [(0.0, "#0d2050"), (0.30, "#145080"),
     (0.65, "#1e8c64"), (0.85, "#a0c830"), (1.0, "#fff014")]
)

# Vista previa
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(LANDSCAPE, origin="lower", cmap=CMAP, extent=[0, 100, 0, 100])
plt.colorbar(im, ax=ax, label="Concentración normalizada")
ax.scatter([75], [75], s=180, facecolors="none", edgecolors="white", lw=2,
           zorder=5, label="Fuente principal (meta)")
ax.scatter([22], [30], s=110, facecolors="none", edgecolors="white", lw=1.5,
           alpha=0.7, zorder=5, label="Máximo local (trampa)")
ax.legend(fontsize=9, loc="upper left")
ax.set_title("Paisaje de concentración de nutrientes", fontsize=12)
ax.set_xlabel("x"); ax.set_ylabel("y")
plt.tight_layout(); plt.show()
print("El punto blanco grande es la meta. El punto pequeño es la trampa.")


## Celda 3 — Simulación del agente

Implementa la ecuación del capítulo exactamente como aparece en el texto.

In [ ]:
# ── Simulación del agente ──────────────────────────────────────────────────

def sample(landscape, x, y, world_size=WORLD_SIZE):
    xi = int(np.clip(x, 0, world_size - 1))
    yi = int(np.clip(y, 0, world_size - 1))
    return landscape[yi, xi]


def run_simulation(a, b, noise_level, n_steps=1200, seed=None):
    """
    Simula la bacteria siguiendo la ecuación de ascenso de colina.

    Parámetros
    ----------
    a           : adaptación sensorial  (0 < a < 1)
    b           : sensibilidad al cambio (b > 0)
    noise_level : nivel de ruido en la detección (0 = sin ruido)
    n_steps     : número de pasos
    seed        : semilla aleatoria (reproducibilidad)

    Retorna
    -------
    dict con listas: x, y, Y, conc, mode
    """
    if seed is not None:
        np.random.seed(seed)

    x, y = 10.0, 10.0          # posición inicial (esquina inferior izquierda)
    angle0 = np.random.uniform(0, 2 * np.pi)
    SPEED = 2.5
    vx = np.cos(angle0) * SPEED
    vy = np.sin(angle0) * SPEED

    Y = 0.0
    prev_x, prev_y = x, y
    THRESHOLD = 0.0

    hist = {"x": [x], "y": [y], "Y": [Y],
            "conc": [sample(LANDSCAPE, x, y)], "mode": ["explore"]}

    tumble_count = 0

    for t in range(n_steps):
        X_now  = sample(LANDSCAPE, x, y)
        X_prev = sample(LANDSCAPE, prev_x, prev_y)

        noise = np.random.normal(0, noise_level * 0.3)

        # ── Ecuación del capítulo ─────────────────────────────────────────
        dX = (X_now - X_prev) + noise
        Y  = a * Y + b * dX
        # ─────────────────────────────────────────────────────────────────

        if Y > THRESHOLD:
            mode = "exploit"
            # Nado recto con pequeña perturbación biológica
            wobble = np.random.normal(0, 0.15)
            angle  = np.arctan2(vy, vx) + wobble
            vx = np.cos(angle) * SPEED
            vy = np.sin(angle) * SPEED
        else:
            mode = "explore"
            tumble_count += 1
            if tumble_count % 8 == 0:       # nueva dirección cada 8 pasos
                angle = np.random.uniform(0, 2 * np.pi)
                vx = np.cos(angle) * SPEED
                vy = np.sin(angle) * SPEED

        prev_x, prev_y = x, y
        x = float(np.clip(x + vx, 1, WORLD_SIZE - 2))
        y = float(np.clip(y + vy, 1, WORLD_SIZE - 2))
        if x <= 1 or x >= WORLD_SIZE - 2: vx = -vx
        if y <= 1 or y >= WORLD_SIZE - 2: vy = -vy

        hist["x"].append(x); hist["y"].append(y)
        hist["Y"].append(Y)
        hist["conc"].append(sample(LANDSCAPE, x, y))
        hist["mode"].append(mode)

    return hist


# Prueba rápida
h0 = run_simulation(a=0.88, b=0.70, noise_level=0.20, n_steps=800, seed=42)
exploit_pct = sum(1 for m in h0["mode"] if m == "exploit") / len(h0["mode"]) * 100
print(f"Simulación completada: {len(h0['x'])} pasos")
print(f"Concentración inicial: {h0['conc'][0]:.3f}")
print(f"Concentración final:   {h0['conc'][-1]:.3f}")
print(f"Máxima alcanzada:      {max(h0['conc']):.3f}")
print(f"% tiempo explotando:   {exploit_pct:.1f}%")


## Celda 4 — Visualización

In [ ]:
# ── Función de visualización ───────────────────────────────────────────────

def plot_simulation(hist, a, b, noise_level, title=None):
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.patch.set_facecolor("#0d1117")
    for ax in axes:
        ax.set_facecolor("#161b22")
        ax.tick_params(colors="#8b949e")
        for sp in ax.spines.values():
            sp.set_edgecolor("#30363d")

    xs    = np.array(hist["x"])
    ys    = np.array(hist["y"])
    Ys    = np.array(hist["Y"])
    cs    = np.array(hist["conc"])
    modes = hist["mode"]
    n     = len(xs)

    # ── Panel 1: Trayectoria ───────────────────────────────────────────────
    ax = axes[0]
    ax.imshow(LANDSCAPE, origin="lower", cmap=CMAP,
              extent=[0, 100, 0, 100], alpha=0.85)
    for i in range(1, n):
        alpha = 0.25 + 0.75 * (i / n)
        if modes[i] == "exploit":
            ax.plot([xs[i-1], xs[i]], [ys[i-1], ys[i]],
                    color="#ff7832", alpha=alpha, lw=1.8)
        else:
            ax.plot([xs[i-1], xs[i]], [ys[i-1], ys[i]],
                    color="#7ed6f8", alpha=alpha * 0.65, lw=1.1)

    ax.scatter(xs[0],  ys[0],  s=80,  c="white",   zorder=6, label="Inicio")
    ax.scatter(xs[-1], ys[-1], s=160, c="#ff7832",
               marker="*", zorder=6, label="Final")
    ax.scatter([75], [75], s=220, facecolors="none", edgecolors="white",
               lw=2, zorder=5, label="Fuente principal")
    ax.scatter([22], [30], s=130, facecolors="none", edgecolors="white",
               lw=1.5, alpha=0.7, zorder=5, label="Máximo local")

    p1 = mpatches.Patch(color="#ff7832", label="Explotación")
    p2 = mpatches.Patch(color="#7ed6f8", label="Exploración")
    ax.legend(handles=[p1, p2], fontsize=8, loc="lower right",
              framealpha=0.3, labelcolor="white", facecolor="#0d1117")
    ax.set_title("Trayectoria", color="#c9d1d9", fontsize=12)
    ax.set_xlabel("x", color="#8b949e")
    ax.set_ylabel("y", color="#8b949e")

    # ── Panel 2: Y(t) ──────────────────────────────────────────────────────
    ax = axes[1]
    t_ax = np.arange(n)
    ax.axhline(0, color="#4ac", lw=1, ls="--", alpha=0.8, label="Umbral (0)")
    mask = Ys > 0
    ax.fill_between(t_ax, Ys, 0, where=mask,   color="#ff7832", alpha=0.3)
    ax.fill_between(t_ax, Ys, 0, where=~mask,  color="#7ed6f8", alpha=0.3)
    ax.plot(t_ax, Ys, color="#e6edf3", lw=0.8, alpha=0.9)
    ax.set_title("Variable de decisión Y(t)", color="#c9d1d9", fontsize=12)
    ax.set_xlabel("Paso", color="#8b949e")
    ax.set_ylabel("Y(t)", color="#8b949e")
    ax.legend(fontsize=9, framealpha=0.3, labelcolor="white", facecolor="#0d1117")
    ax.text(0.02, 0.97, f"Y(t+1) = {a}·Y(t) + {b}·ΔX",
            transform=ax.transAxes, fontsize=9, color="#7ec8e3", va="top",
            bbox=dict(boxstyle="round", facecolor="#0d1117", alpha=0.7))

    # ── Panel 3: Concentración ─────────────────────────────────────────────
    ax = axes[2]
    ax.plot(t_ax, cs, color="#f0c040", lw=1.2, alpha=0.9)
    ax.axhline(cs.max(), color="white", lw=0.8, ls=":", alpha=0.5)
    ax.text(n * 0.02, cs.max() + 0.01, f"Máx = {cs.max():.2f}",
            color="white", fontsize=8)
    ax.set_title("Concentración detectada X(t)", color="#c9d1d9", fontsize=12)
    ax.set_xlabel("Paso", color="#8b949e")
    ax.set_ylabel("X(t)", color="#8b949e")
    ax.set_ylim(0, 1.08)

    auto_title = title or f"Ascenso de colina  |  a = {a},  b = {b},  ruido = {noise_level}"
    fig.suptitle(auto_title, color="#e6edf3", fontsize=13, y=1.02)
    exploit_pct = mask.sum() / n * 100
    fig.text(0.5, -0.03,
             f"Explotación: {exploit_pct:.0f}%  |  Conc. final: {cs[-1]:.2f}"
             f"  |  Conc. máxima: {cs.max():.2f}",
             ha="center", color="#8b949e", fontsize=10)
    plt.tight_layout(); plt.show()


# Vista rápida con parámetros por defecto
plot_simulation(h0, a=0.88, b=0.70, noise_level=0.20)


## Celda 5 — Simulador interactivo

Ajusta los sliders y presiona **Ejecutar simulación**.
Los experimentos rápidos reproducen los casos de los ejercicios del capítulo.

In [ ]:
# ── Simulador interactivo con sliders ─────────────────────────────────────
# Ajusta los parámetros y presiona "Ejecutar simulación"

style  = {"description_width": "210px"}
layout = widgets.Layout(width="500px")

sl_a     = widgets.FloatSlider(value=0.88, min=0.10, max=0.99, step=0.01,
                               description="a — Adaptación sensorial",
                               style=style, layout=layout, readout_format=".2f")
sl_b     = widgets.FloatSlider(value=0.70, min=0.10, max=2.00, step=0.05,
                               description="b — Sensibilidad al cambio",
                               style=style, layout=layout, readout_format=".2f")
sl_noise = widgets.FloatSlider(value=0.20, min=0.00, max=1.00, step=0.05,
                               description="Ruido ambiental",
                               style=style, layout=layout, readout_format=".2f")
sl_steps = widgets.IntSlider(  value=1000, min=200, max=2000, step=100,
                               description="Número de pasos",
                               style=style, layout=layout)

PRESETS = {
    "— Experimentos del capítulo —":  None,
    "Balance óptimo (defecto)":       dict(a=0.88, b=0.70, noise=0.20),
    "Atrapado en máximo local":       dict(a=0.97, b=0.50, noise=0.05),
    "Exploración excesiva":           dict(a=0.25, b=1.00, noise=0.10),
    "Entorno muy ruidoso":            dict(a=0.70, b=1.50, noise=0.85),
    "Adaptación muy lenta (a≈1)":    dict(a=0.99, b=0.30, noise=0.10),
}

dd_preset = widgets.Dropdown(options=list(PRESETS.keys()),
                              description="Experimento rápido:",
                              style={"description_width": "160px"},
                              layout=widgets.Layout(width="500px"))

btn_run   = widgets.Button(description="▶  Ejecutar simulación",
                           button_style="success",
                           layout=widgets.Layout(width="230px"))
btn_reset = widgets.Button(description="↺  Parámetros por defecto",
                           button_style="warning",
                           layout=widgets.Layout(width="230px"))
out = widgets.Output()

def run_and_plot(_=None):
    with out:
        clear_output(wait=True)
        h = run_simulation(a=sl_a.value, b=sl_b.value,
                           noise_level=sl_noise.value,
                           n_steps=sl_steps.value, seed=42)
        plot_simulation(h, a=sl_a.value, b=sl_b.value,
                        noise_level=sl_noise.value)

def reset_params(_=None):
    sl_a.value = 0.88; sl_b.value = 0.70
    sl_noise.value = 0.20; sl_steps.value = 1000
    dd_preset.value = "— Experimentos del capítulo —"

def apply_preset(change):
    p = PRESETS.get(change["new"])
    if p:
        sl_a.value = p["a"]; sl_b.value = p["b"]; sl_noise.value = p["noise"]

btn_run.on_click(run_and_plot)
btn_reset.on_click(reset_params)
dd_preset.observe(apply_preset, names="value")

display(widgets.VBox([sl_a, sl_b, sl_noise, sl_steps, dd_preset,
                      widgets.HBox([btn_run, btn_reset])]), out)
run_and_plot()        # ejecutar con valores por defecto al abrir


## Ejercicios del capítulo

Estas celdas corresponden a los ejercicios de comprensión del Capítulo 4.
Modifica los parámetros y responde las preguntas al final de cada celda.

### Ejercicio 3 — Cálculo manual de Y(t)

In [ ]:
# ── Ejercicio 3 del capítulo ───────────────────────────────────────────────
# Ecuación: Y(t+1) = a·Y(t) + b·[X(t+1) - X(t)]
# Parámetros: a = 0.8, b = 1.0, umbral = 0
# Secuencia de concentraciones: X = [10, 12, 14, 14, 13]

a_ej, b_ej = 0.8, 1.0
X_seq = [10, 12, 14, 14, 13]

print(f"Parámetros: a = {a_ej}, b = {b_ej}, umbral = 0")
print(f"{'t':<6} {'X(t)':<8} {'ΔX':<8} {'Y(t)':<10} Modo")
print("─" * 46)

Y = 0.0
for t in range(len(X_seq)):
    if t == 0:
        print(f"{t:<6} {X_seq[t]:<8.1f} {'—':<8} {Y:<10.3f} — (inicial)")
        continue
    dX = X_seq[t] - X_seq[t-1]
    Y  = a_ej * Y + b_ej * dX
    modo = "Explotar ▶" if Y > 0 else "Explorar ↺"
    print(f"{t:<6} {X_seq[t]:<8.1f} {dX:<+8.1f} {Y:<10.3f} {modo}")

print()
print("Preguntas:")
print("  1. ¿En qué paso cambia de modo el agente?")
print("  2. ¿Por qué Y sigue positivo en t=3 aunque la concentración no cambia?")
print("  3. ¿Qué pasaría con a = 1.0 exactamente?")


### Ejercicio 5 — Máximos locales y adaptación sensorial

In [ ]:
# ── Ejercicio 5: máximos locales y adaptación sensorial ───────────────────
# Compara dos valores de 'a': uno muy alto (casi sin adaptación)
# y uno moderado. ¿Se queda atrapada la bacteria en el máximo local?

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
fig.patch.set_facecolor("#0d1117")

configs = [
    dict(a=0.97, b=0.50, noise=0.05, label="a = 0.97  (adaptación muy lenta)"),
    dict(a=0.88, b=0.70, noise=0.15, label="a = 0.88  (balance moderado)"),
]

for ax, cfg in zip(axes, configs):
    ax.set_facecolor("#161b22")
    ax.imshow(LANDSCAPE, origin="lower", cmap=CMAP,
              extent=[0, 100, 0, 100], alpha=0.85)

    h = run_simulation(a=cfg["a"], b=cfg["b"],
                       noise_level=cfg["noise"], n_steps=1200, seed=42)
    xs, ys, modes = np.array(h["x"]), np.array(h["y"]), h["mode"]
    n = len(xs)

    for i in range(1, n):
        alpha = 0.25 + 0.75 * (i / n)
        color = "#ff7832" if modes[i] == "exploit" else "#7ed6f8"
        ax.plot([xs[i-1], xs[i]], [ys[i-1], ys[i]], color=color,
                alpha=alpha, lw=1.5)

    ax.scatter([75], [75], s=220, facecolors="none", edgecolors="white", lw=2, zorder=5)
    ax.scatter([22], [30], s=130, facecolors="none", edgecolors="white", lw=1.5,
               alpha=0.7, zorder=5)
    ax.scatter(xs[0],  ys[0],  s=80,  c="white",   zorder=6)
    ax.scatter(xs[-1], ys[-1], s=160, c="#ff7832",  marker="*", zorder=6)

    max_c = max(h["conc"])
    trap_time = sum(1 for xi, yi in zip(xs, ys)
                    if (xi-22)**2 + (yi-30)**2 < 15**2)
    ax.set_title(f"{cfg['label']}\nConc. máx = {max_c:.2f}  |  "
                 f"Pasos cerca de la trampa: {trap_time}",
                 color="#c9d1d9", fontsize=10)
    for sp in ax.spines.values(): sp.set_edgecolor("#30363d")
    ax.tick_params(colors="#8b949e")
    ax.set_xlabel("x", color="#8b949e"); ax.set_ylabel("y", color="#8b949e")

plt.suptitle("¿La bacteria escapa del máximo local?", color="#e6edf3",
             fontsize=13, y=1.02)
plt.tight_layout(); plt.show()
print("¿Con cuál valor de a la bacteria escapa del máximo local?")
print("¿Por qué la adaptación sensorial es necesaria para escapar?")


### Ejercicio libre — Búsqueda de parámetros óptimos bajo ruido

In [ ]:
# ── Ejercicio libre: búsqueda de parámetros óptimos bajo ruido ─────────────
# ¿Qué combinación de a y b produce la mayor concentración final
# en un entorno muy ruidoso (noise = 0.8)?

resultados = []
a_vals = [0.5, 0.70, 0.88, 0.97]
b_vals = [0.3, 0.7,  1.2,  1.8]

for a_val in a_vals:
    for b_val in b_vals:
        h = run_simulation(a=a_val, b=b_val, noise_level=0.8,
                           n_steps=800, seed=42)
        resultados.append(dict(a=a_val, b=b_val,
                               max_conc=max(h["conc"]),
                               final_conc=h["conc"][-1]))

resultados.sort(key=lambda r: r["max_conc"], reverse=True)

print(f"{'a':>6}  {'b':>6}  {'Conc. máx':>12}  {'Conc. final':>12}")
print("─" * 46)
for r in resultados:
    marker = "  ← mejor" if r == resultados[0] else ""
    print(f"{r['a']:>6.2f}  {r['b']:>6.2f}  {r['max_conc']:>12.3f}"
          f"  {r['final_conc']:>12.3f}{marker}")

print()
print("Reflexión:")
print("  ¿Por qué valores de b altos fallan con mucho ruido?")
print("  ¿Qué relación hay entre a óptimo y la tasa de adaptación sensorial?")


## Créditos y licencia

Este notebook es parte del proyecto:

> **Bouzas, A. (2026).** *Aprendizaje y Comportamiento Adaptable: Principios y Modelos.*  
> Lab25, Facultad de Psicología, UNAM.  
> https://www.bouzaslab25.com

Código disponible en: **https://github.com/bouzaslab25/libro-aca**  
Licencia: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)
